# PIPELINE 03: CONTENT BEHAVIOR ANALYSIS (TESTING & PROTOTYPING)
**Mục tiêu:** Dự án này tập trung vào việc xử lý dữ liệu xem phim (log_content) để trích xuất các chỉ số quan trọng: Số ngày hoạt động (Active Days), Sở thích (Taste) và Thể loại xem nhiều nhất (MostWatch).

#### 1. Khởi tạo môi trường & Cấu hình
Ở bước này, chúng ta nạp các thư viện cần thiết và định nghĩa Schema để ép Spark đọc dữ liệu JSON một cách chuẩn xác nhất.

In [1]:
!pip install findspark

In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import functions as F
import os

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("Test_Content_Pipeline") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Cấu hình đường dẫn (Hãy điều chỉnh cho đúng máy bạn)
BASE_DIR = "/Users/hoaithuong/Desktop/Customer-Behavior-Analysis-Pipeline"
RAW_CONTENT_PATH = "/Users/hoaithuong/Desktop/DE CLASS/Dataset/log_content"
# Định nghĩa Schema
CONTENT_SCHEMA = StructType([
    StructField("_source", StructType([
        StructField("Contract", StringType(), True),
        StructField("AppName", StringType(), True),
        StructField("TotalDuration", LongType(), True)
    ]), True)
])

print(">>> Spark Session đã sẵn sàng!")

26/04/01 18:29:49 WARN Utils: Your hostname, Ryms.local resolves to a loopback address: 127.0.0.1; using 172.16.2.25 instead (on interface en0)
26/04/01 18:29:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 18:29:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


>>> Spark Session đã sẵn sàng!


#### 2. Đọc dữ liệu thô (Raw Data)
Chúng ta sẽ đọc toàn bộ các file JSON và trích xuất ngày tháng từ tên file (giả định tên file có dạng YYYYMMDD.json).

In [3]:
print(">>> Reading Data...")
try:
    raw_df = spark.read.schema(CONTENT_SCHEMA).json(f"{RAW_CONTENT_PATH}/*.json")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

>>> Reading Data...


#### 3. Làm sạch & Phân loại (Cleaning & Mapping)
Chuyển đổi các AppName vụn vặt thành 5 nhóm Category chính: Phim Truyện, Truyền Hình, Giải Trí, Thiếu Nhi, Thể Thao. 

In [4]:
df_clean = raw_df.withColumn("file_name", input_file_name()) \
        .withColumn("DateStr", regexp_extract(col("file_name"), r"(\d{8})\.json$", 1)) \
        .withColumn("Date", to_date(col("DateStr"), "yyyyMMdd")) \
        .drop("file_name", "DateStr")

In [5]:
df_mapped = df_clean.select(
        F.col("_source.Contract").alias("user_id"),
        F.col("_source.TotalDuration").cast("long").alias("TotalDuration"),
        F.col("Date"),
        F.when(F.col("_source.AppName").isin('CHANNEL', 'DSHD', 'KPLUS', 'KPlus'), "Truyền Hình")
        .when(F.col("_source.AppName").isin('VOD', 'FIMS_RES', 'BHD_RES', 'VOD_RES', 'FIMS', 'BHD', 'DANET'), "Phim Truyện")
        .when(F.col("_source.AppName") == 'RELAX', "Giải Trí")
        .when(F.col("_source.AppName") == 'CHILD', "Thiếu Nhi")
        .when(F.col("_source.AppName") == 'SPORT', "Thể Thao")
        .otherwise("Other").alias("Type") 
    )

# Lọc rác
df_clean = df_mapped.filter(
    (F.col("TotalDuration") > 0) & 
    (F.col("Type") != "Other") & 
    (F.col("user_id").rlike("^[a-zA-Z0-9]")) 
).cache()

print("Dữ liệu sau khi làm sạch và phân loại:")
df_clean.show(10, truncate=False)

Dữ liệu sau khi làm sạch và phân loại:


+---------+-------------+----------+-----------+
|user_id  |TotalDuration|Date      |Type       |
+---------+-------------+----------+-----------+
|HNH579912|254          |2022-04-01|Truyền Hình|
|HUFD40665|1457         |2022-04-01|Truyền Hình|
|HNH572635|2318         |2022-04-01|Truyền Hình|
|HND141717|1452         |2022-04-01|Truyền Hình|
|HNH743103|251          |2022-04-01|Truyền Hình|
|HNH893773|924          |2022-04-01|Truyền Hình|
|HND083642|1444         |2022-04-01|Truyền Hình|
|DNFD74404|691          |2022-04-01|Truyền Hình|
|DTFD21200|1436         |2022-04-01|Truyền Hình|
|LDFD05747|1434         |2022-04-01|Truyền Hình|
+---------+-------------+----------+-----------+
only showing top 10 rows



#### 4. Tính toán KPIs (Pivot & Aggregation)
Đây là bước quan trọng nhất: Xoay bảng (Pivot) để mỗi User là một dòng, chứa tổng thời lượng xem của từng thể loại.

In [8]:
# 1. Tính Active Days (Dùng dropDuplicates để tối ưu)
TOTAL_DAYS_IN_MONTH = 30                
df_active = df_clean.select("user_id", "Date").dropDuplicates() \
    .groupBy("user_id").agg(
        # Phép tính: Số ngày / Tổng số ngày
        F.round(F.count("Date") / TOTAL_DAYS_IN_MONTH, 2).alias("Active")
    )

# 2. Pivot Duration
categories = ["Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao"]
df_pivot = df_clean.groupBy("user_id").pivot("Type", categories).sum("TotalDuration").fillna(0)

# 3. Tìm MostWatch & Taste
df_result = df_pivot.withColumn("max_val", F.greatest(*[F.col(c) for c in categories]))
most_watch_expr = F.when(F.col("max_val") == 0, "None")
for cat in categories:
    most_watch_expr = most_watch_expr.when(F.col("max_val") == F.col(cat), cat)

final_df = df_result.withColumn("MostWatch", most_watch_expr.otherwise("Mixed")) \
                    .withColumn("Taste", F.concat_ws(" - ", *[F.when(F.col(c) > 0, F.lit(c)) for c in categories])) \
                    .join(df_active, on="user_id", how="inner") \
                    .drop("max_val")

# Đổi tên cột cho đẹp
for cat in categories:
    final_df = final_df.withColumnRenamed(cat, "Total_" + cat.replace(" ", "_"))

print("KẾT QUẢ CUỐI CÙNG (MASTER CONTENT ENRICHED):")
final_df.show(10)

KẾT QUẢ CUỐI CÙNG (MASTER CONTENT ENRICHED):


26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:34:19 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
|       user_id|Total_Truyền_Hình|Total_Phim_Truyện|Total_Giải_Trí|Total_Thiếu_Nhi|Total_Thể_Thao|  MostWatch|               Taste|Active|
+--------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
|113.182.209.48|               63|                0|            89|              0|             0|   Giải Trí|Truyền Hình - Giả...|  0.03|
|14.182.110.125|              404|                0|            92|              0|             0|Truyền Hình|Truyền Hình - Giả...|  0.03|
|     AGAAA0338|           278633|                0|             0|              0|             0|Truyền Hình|         Truyền Hình|   1.0|
|     AGAAA0342|           117788|                0|           204|              0|             0|Truyền Hình|Truyền Hình - Giả...|   0.4|
|     AGAAA0346|          2

26/04/02 06:49:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1047827 ms exceeds timeout 120000 ms
26/04/02 06:49:24 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/02 07:06:12 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

In [ ]:
final_df.show(10)

26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/01 18:23:50 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+-----------+
|       user_id|Total_Truyền_Hình|Total_Phim_Truyện|Total_Giải_Trí|Total_Thiếu_Nhi|Total_Thể_Thao|  MostWatch|               Taste|Active_Days|
+--------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+-----------+
|113.182.209.48|               63|                0|            89|              0|             0|   Giải Trí|Truyền Hình - Giả...|          1|
|14.182.110.125|              404|                0|            92|              0|             0|Truyền Hình|Truyền Hình - Giả...|          1|
|     AGAAA0338|           278633|                0|             0|              0|             0|Truyền Hình|         Truyền Hình|         30|
|     AGAAA0342|           117788|                0|           204|              0|             0|Truyền Hình|Truyền Hình - Giả...|     

In [ ]:
final_df.count()

1920510

#### 5. Lưu trữ kết quả (Export)
Lưu kết quả ra định dạng Parquet để sẵn sàng cho bước JOIN với Search ở Pipeline tiếp theo.

In [ ]:
# OUTPUT_PATH = f"{BASE_DIR}/data/processed/Master_Content_Enriched.parquet"

# final_df.write.mode("overwrite").parquet(OUTPUT_PATH)
# print(f"✅ Đã lưu file thành công tại: {OUTPUT_PATH}")